# LeetCode 530 — Minimum Absolute Difference in BST bằng Python

**Link đề:** https://leetcode.com/problems/minimum-absolute-difference-in-bst/
**Độ khó:** Easy · **Chủ đề:** Tree, Depth-First Search, Breadth-First Search, Binary Search Tree, Binary Tree

```python
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def getMinimumDifference(self, root: Optional[TreeNode]) -> int:
        ...
```

## Mục tiêu học
1. Ôn **cây nhị phân tìm kiếm (BST)** và tự dựng `TreeNode`, `build_tree`, `show_tree` để chạy độc lập.
2. Cách trực giác: so **mọi cặp** node — O(n²) — rồi cải tiến bằng **sắp xếp + so phần tử kề nhau**.
3. Insight then chốt: **in-order của BST cho dãy tăng** → hiệu nhỏ nhất chắc chắn nằm giữa **hai phần tử liền kề**.
4. Lời giải chính: in-order với biến `prev`, cài đặt **lặp bằng stack** (an toàn với cây sâu 10⁴).
5. Trace, chứng minh, so sánh 5 cách, luyện biến thể (LeetCode 783, mảng thường, top-k cặp gần nhau).

## 1. Đề bài

Cho `root` của một **BST**, trả về **hiệu tuyệt đối nhỏ nhất** giữa giá trị của **hai node khác nhau bất kỳ** trong cây.

| root (level order) | Output | Giải thích |
|---|---:|---|
| `[4,2,6,1,3]` | 1 | ví dụ `|2 − 1| = 1` |
| `[1,0,48,null,null,12,49]` | 1 | `|49 − 48| = 1` |

```text
Ví dụ 1:      4             Ví dụ 2:     1
            /   \                       / \
           2     6                     0   48
          / \                              / \
         1   3                           12   49
```

**Constraints:** số node trong `[2, 10^4]`; `0 <= Node.val <= 10^5`.

**Ghi chú:** bài này giống hệt LeetCode 783 (Minimum Distance Between BST Nodes).

## 2. Nền tảng: BST, `TreeNode`, dựng và in cây

**BST** (Binary Search Tree): với mọi node, mọi giá trị ở cây con **trái nhỏ hơn**, mọi giá trị ở cây con **phải lớn hơn** giá trị node (bài này giá trị khác nhau).

LeetCode mô tả cây bằng list **level order**, `null` (Python: `None`) là chỗ trống. Ta tự viết hàm dựng cây bằng **hàng đợi** `deque` và hàm in cây nằm ngang.

In [1]:
from collections import deque
from typing import Optional

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i]); queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i]); queue.append(node.right)
        i += 1
    return root

def show_tree(node, depth=0):
    # In nằm ngang: cây con phải ở trên, trái ở dưới (nghiêng đầu sang trái để đọc).
    if node is None:
        return
    show_tree(node.right, depth + 1)
    print("     " * depth + str(node.val))
    show_tree(node.left, depth + 1)

show_tree(build_tree([4, 2, 6, 1, 3]))
print("-" * 20)
show_tree(build_tree([1, 0, 48, None, None, 12, 49]))

     6
4
          3
     2
          1
--------------------
          49
     48
          12
1
     0


`Optional[TreeNode]` (từ module `typing`) nghĩa là "một `TreeNode` hoặc `None`" — chỉ là **chú thích kiểu** (type hint), không ảnh hưởng khi chạy.

### 2.1. Duyệt cây: pre-order và in-order (đệ quy)

- **Pre-order**: node → trái → phải.
- **In-order**: trái → node → phải. Với BST, in-order cho dãy **tăng dần**.

In [2]:
def preorder_values(node, out):
    if node is None:
        return out
    out.append(node.val)
    preorder_values(node.left, out)
    preorder_values(node.right, out)
    return out

def inorder_values(node, out):
    if node is None:
        return out
    inorder_values(node.left, out)
    out.append(node.val)
    inorder_values(node.right, out)
    return out

t2 = build_tree([1, 0, 48, None, None, 12, 49])
print("pre-order:", preorder_values(t2, []))
print("in-order :", inorder_values(t2, []))

pre-order: [1, 0, 48, 12, 49]
in-order : [0, 1, 12, 48, 49]


### 2.2. `abs`, `min`, `float('inf')`, `zip`

- `abs(x)`: giá trị tuyệt đối.
- Khởi tạo "nhỏ nhất" bằng `float('inf')` (vô cực dương) để phép `min` đầu tiên luôn cập nhật.
- `zip(arr, arr[1:])` ghép từng cặp **phần tử kề nhau**.

In [3]:
print(abs(3 - 7), float("inf") > 10**18)
arr = [0, 1, 12, 48, 49]
print(list(zip(arr, arr[1:])))
print(min(b - a for a, b in zip(arr, arr[1:])))

4 True
[(0, 1), (1, 12), (12, 48), (48, 49)]
1


## 3. Cách 1 — Trực giác: thử mọi cặp node

Thu mọi giá trị vào list (thứ tự nào cũng được), so **mọi cặp** `i < j`.

In [4]:
def min_diff_all_pairs(root):
    vals = preorder_values(root, [])
    best = float("inf")
    for i in range(len(vals)):
        for j in range(i + 1, len(vals)):
            best = min(best, abs(vals[i] - vals[j]))
    return best

print(min_diff_all_pairs(build_tree([4, 2, 6, 1, 3])))                 # 1
print(min_diff_all_pairs(build_tree([1, 0, 48, None, None, 12, 49])))  # 1

1
1


### Nút thắt

`n = 10^4` node → `n(n−1)/2 ≈ 5·10^7` cặp: quá chậm trong Python. Và ta **chưa dùng** tính chất BST.

Quan sát: nếu các giá trị đã **sắp xếp** `v0 < v1 < ... < v(n-1)`, thì với `i < j`, `v[j] − v[i] ≥ v[i+1] − v[i]`. Tức là cặp gần nhau nhất luôn là **hai phần tử kề nhau** trong dãy sắp xếp → chỉ cần so `n − 1` cặp.

## 4. Cách 2 — Sắp xếp rồi so phần tử kề nhau (đúng cho mọi cây)

In [5]:
def min_diff_sort(root):
    vals = sorted(preorder_values(root, []))
    return min(b - a for a, b in zip(vals, vals[1:]))

print(min_diff_sort(build_tree([4, 2, 6, 1, 3])))
print(min_diff_sort(build_tree([1, 0, 48, None, None, 12, 49])))

1
1


Đã `O(n log n)`. Nhưng với BST ta **không cần sắp xếp**: in-order đã cho sẵn dãy tăng!

## 5. Insight — in-order + so với phần tử ngay trước (`prev`)

Duyệt in-order, giữ `prev` = giá trị node được thăm **ngay trước** (cũng chính là phần tử kề bên trái trong dãy sắp xếp). Ở mỗi node:
```text
nếu prev tồn tại: best = min(best, node.val - prev)
prev = node.val
```
Không cần abs vì dãy tăng → `node.val - prev > 0`.

### Cách 3 — in-order ra list rồi so kề nhau

In [6]:
def min_diff_inorder_list(root):
    vals = inorder_values(root, [])
    return min(b - a for a, b in zip(vals, vals[1:]))

print(min_diff_inorder_list(build_tree([4, 2, 6, 1, 3])))

1


## 6. Cách 4 — Lời giải chính: in-order với `prev`, không lưu list

### 6.1. Bản đệ quy (dễ đọc)

Dùng `nonlocal` để hàm con gán lại biến của hàm ngoài.

In [7]:
def min_diff_inorder_recursive(root):
    prev = None
    best = float("inf")

    def dfs(node):
        nonlocal prev, best
        if node is None:
            return
        dfs(node.left)
        if prev is not None:
            best = min(best, node.val - prev)
        prev = node.val
        dfs(node.right)

    dfs(root)
    return best

print(min_diff_inorder_recursive(build_tree([4, 2, 6, 1, 3])))
print(min_diff_inorder_recursive(build_tree([1, 0, 48, None, None, 12, 49])))

1
1


Lưu ý: dùng `prev is not None` chứ **không** `if prev:` — vì giá trị node có thể là `0` (falsy) như ví dụ 2!

### 6.2. Bản lặp bằng stack (an toàn với cây sâu)

In-order lặp:
1. Đi trái hết mức, đẩy các node vào stack.
2. Pop một node → **thăm** nó (so với `prev`).
3. Chuyển sang cây con phải, lặp lại.

In [8]:
def min_diff_inorder_iterative(root):
    stack, cur = [], root
    prev, best = None, float("inf")
    while stack or cur:
        while cur:                   # 1. đi trái hết mức
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()            # 2. node nhỏ nhất chưa thăm
        if prev is not None:
            best = min(best, cur.val - prev)
        prev = cur.val
        cur = cur.right              # 3. sang phải
    return best

print(min_diff_inorder_iterative(build_tree([4, 2, 6, 1, 3])))
print(min_diff_inorder_iterative(build_tree([1, 0, 48, None, None, 12, 49])))

1
1


## 7. TRACE từng bước

Cây ví dụ 2 `[1, 0, 48, null, null, 12, 49]`, in-order `0, 1, 12, 48, 49`:

| bước | stack trước khi pop | thăm | prev trước | hiệu | best sau |
|---:|---|---:|---|---:|---:|
| 1 | `[1, 0]` | 0 | None | — | inf |
| 2 | `[1]` | 1 | 0 | 1 | 1 |
| 3 | `[48, 12]` | 12 | 1 | 11 | 1 |
| 4 | `[48]` | 48 | 12 | 36 | 1 |
| 5 | `[49]` | 49 | 48 | 1 | 1 |

In [9]:
def trace_min_diff(root):
    stack, cur = [], root
    prev, best, step = None, float("inf"), 0
    while stack or cur:
        while cur:
            stack.append(cur)
            cur = cur.left
        step += 1
        snapshot = [n.val for n in stack]
        cur = stack.pop()
        if prev is not None:
            diff = cur.val - prev
            best = min(best, diff)
            info = f"hiệu={diff:<3}"
        else:
            info = "hiệu=—  "
        print(f"bước {step}: stack={str(snapshot):10s} thăm {cur.val:>3} | prev={str(prev):>4} | {info} best={best}")
        prev = cur.val
        cur = cur.right
    return best

trace_min_diff(build_tree([1, 0, 48, None, None, 12, 49]))

bước 1: stack=[1, 0]     thăm   0 | prev=None | hiệu=—   best=inf
bước 2: stack=[1]        thăm   1 | prev=   0 | hiệu=1   best=1
bước 3: stack=[48, 12]   thăm  12 | prev=   1 | hiệu=11  best=1
bước 4: stack=[48]       thăm  48 | prev=  12 | hiệu=36  best=1
bước 5: stack=[49]       thăm  49 | prev=  48 | hiệu=1   best=1


1

Trace ví dụ 1 (`[4,2,6,1,3]`, in-order `1, 2, 3, 4, 6`):

In [10]:
trace_min_diff(build_tree([4, 2, 6, 1, 3]))

bước 1: stack=[4, 2, 1]  thăm   1 | prev=None | hiệu=—   best=inf
bước 2: stack=[4, 2]     thăm   2 | prev=   1 | hiệu=1   best=1
bước 3: stack=[4, 3]     thăm   3 | prev=   2 | hiệu=1   best=1
bước 4: stack=[4]        thăm   4 | prev=   3 | hiệu=1   best=1
bước 5: stack=[6]        thăm   6 | prev=   4 | hiệu=2   best=1


1

### Phản ví dụ cho một lối tắt sai: "chỉ so node với con trực tiếp"

Nhiều bạn chỉ so `|node − node.left|` và `|node − node.right|`. Nhưng cặp gần nhất có thể **không phải cha–con**! Hãy chạy thử với cây `[4, 1, 9, null, 3, 7]`:

In [11]:
def wrong_parent_child(root):
    best = float("inf")
    stack = [root]
    while stack:
        node = stack.pop()
        for child in (node.left, node.right):
            if child:
                best = min(best, abs(node.val - child.val))
                stack.append(child)
    return best

t = build_tree([4, 1, 9, None, 3, 7])      # in-order: 1, 3, 4, 7, 9
show_tree(t)
print("chỉ so cha–con:", wrong_parent_child(t), "| đúng:", min_diff_inorder_iterative(t))

     9
          7
4
          3
     1
chỉ so cha–con: 2 | đúng: 1


Ở đây cặp `(3, 4)` có hiệu 1 nhưng `3` là **cháu** của `4` (con phải của `1`), không phải con trực tiếp. Cách so cha–con trả `2` (sai).

## 8. Cách 5 (nâng cao) — Morris in-order, bộ nhớ phụ O(1)

Giống bài 501: tạm nối node phải nhất của cây con trái về node hiện tại để quay lại mà không cần stack; gỡ nối sau khi dùng.

In [12]:
def min_diff_morris(root):
    prev, best = None, float("inf")
    cur = root
    while cur:
        if cur.left is None:
            if prev is not None:
                best = min(best, cur.val - prev)
            prev = cur.val
            cur = cur.right
        else:
            pre = cur.left
            while pre.right is not None and pre.right is not cur:
                pre = pre.right
            if pre.right is None:
                pre.right = cur
                cur = cur.left
            else:
                pre.right = None
                if prev is not None:
                    best = min(best, cur.val - prev)
                prev = cur.val
                cur = cur.right
    return best

t = build_tree([1, 0, 48, None, None, 12, 49])
print(min_diff_morris(t), "| cây còn nguyên:", inorder_values(t, []))

1 | cây còn nguyên: [0, 1, 12, 48, 49]


## 9. Code nộp LeetCode

### Bản in-order lặp (khuyên dùng)

In [13]:
class Solution:
    def getMinimumDifference(self, root: Optional[TreeNode]) -> int:
        stack, cur = [], root
        prev, best = None, float("inf")
        while stack or cur:
            while cur:
                stack.append(cur)
                cur = cur.left
            cur = stack.pop()
            if prev is not None:
                best = min(best, cur.val - prev)
            prev = cur.val
            cur = cur.right
        return best

s = Solution()
print(s.getMinimumDifference(build_tree([4, 2, 6, 1, 3])))                 # 1
print(s.getMinimumDifference(build_tree([1, 0, 48, None, None, 12, 49])))  # 1

1
1


### Bản in-order đệ quy (ngắn, dễ nói trong phỏng vấn)

In [14]:
class SolutionRecursive:
    def getMinimumDifference(self, root: Optional[TreeNode]) -> int:
        self.prev = None
        self.best = float("inf")

        def dfs(node):
            if not node:
                return
            dfs(node.left)
            if self.prev is not None:
                self.best = min(self.best, node.val - self.prev)
            self.prev = node.val
            dfs(node.right)

        dfs(root)
        return self.best

print(SolutionRecursive().getMinimumDifference(build_tree([4, 2, 6, 1, 3])))

1


(Bản đệ quy dùng thuộc tính `self.prev` thay cho `nonlocal` — một cách khác để chia sẻ trạng thái. Với cây lệch 10⁴ node, đệ quy có thể vượt giới hạn mặc định ~1000 tầng của Python; khi nộp, đổi tên lớp thành `Solution` và cân nhắc `sys.setrecursionlimit` hoặc dùng bản lặp.)

## 10. Vì sao đúng?

1. **In-order của BST là dãy tăng** `v0 < v1 < ... < v(n−1)` (quy nạp: in-order(trái) < node < in-order(phải)).
2. **Bổ đề kề nhau:** với `i < j`, `v[j] − v[i] = (v[j] − v[j−1]) + ... + (v[i+1] − v[i]) ≥ v[i+1] − v[i]` vì mọi số hạng dương. Vậy hiệu nhỏ nhất trong mọi cặp bằng hiệu nhỏ nhất trong các cặp **kề nhau**.
3. **Invariant** của vòng lặp: sau khi thăm `k` node đầu của in-order, `prev = v(k−1)` và `best = min(v[t] − v[t−1])` với `1 ≤ t ≤ k−1`.
4. Kết thúc (`k = n ≥ 2`): `best` là hiệu nhỏ nhất của mọi cặp kề nhau ⇒ đáp án đúng. (`n ≥ 2` nên `best` không còn là `inf`.)

## 11. Độ phức tạp

`n` = số node, `h` = chiều cao (`log n ≤ h ≤ n`).

| Cách | Time | Space phụ | Ghi chú |
|---|---:|---:|---|
| Mọi cặp | `O(n²)` | `O(n)` | quá chậm với n = 10^4 |
| Sắp xếp + kề nhau | `O(n log n)` | `O(n)` | đúng với cả cây thường |
| In-order ra list | `O(n)` | `O(n)` | dùng BST |
| In-order + `prev` (đệ quy/stack) | `O(n)` | `O(h)` | ⭐ chuẩn |
| Morris | `O(n)` | `O(1)` | sửa tạm cây rồi khôi phục |

## 12. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

s = Solution()
check("đề 1", s.getMinimumDifference(build_tree([4, 2, 6, 1, 3])), 1)
check("đề 2 (có node 0)", s.getMinimumDifference(build_tree([1, 0, 48, None, None, 12, 49])), 1)
check("2 node", s.getMinimumDifference(build_tree([5, 1])), 4)
check("2 node (con phải)", s.getMinimumDifference(build_tree([0, None, 100000])), 100000)
check("cặp gần nhất không phải cha–con", s.getMinimumDifference(build_tree([4, 1, 9, None, 3, 7])), 1)
check("lệch trái", s.getMinimumDifference(build_tree([10, 7, None, 3])), 3)

PASS đề 1: 1
PASS đề 2 (có node 0): 1
PASS 2 node: 4
PASS 2 node (con phải): 100000
PASS cặp gần nhất không phải cha–con: 1
PASS lệch trái: 3


In [16]:
import random

def bst_from_values(values):
    # Chèn lần lượt (bằng vòng lặp) để tạo BST với giá trị khác nhau.
    root = None
    for v in values:
        node = TreeNode(v)
        if root is None:
            root = node; continue
        cur = root
        while True:
            if v < cur.val:
                if cur.left is None: cur.left = node; break
                cur = cur.left
            else:
                if cur.right is None: cur.right = node; break
                cur = cur.right
    return root

random.seed(530)
for _ in range(400):
    vals = random.sample(range(0, 300), random.randint(2, 30))
    root = bst_from_values(vals)
    expected = min_diff_all_pairs(root)
    for f in [min_diff_sort, min_diff_inorder_list, min_diff_inorder_recursive,
              min_diff_inorder_iterative, min_diff_morris,
              Solution().getMinimumDifference, SolutionRecursive().getMinimumDifference]:
        assert f(root) == expected, (vals, f.__name__)
print("400 BST ngẫu nhiên: 7 cài đặt khớp brute force mọi cặp — PASS")

# Cây lệch 10^4 node: bản lặp không bị lỗi đệ quy
deep = bst_from_values(list(range(0, 20000, 2)))      # chèn tăng dần -> cây thành đường thẳng
print("cây lệch 10^4 node:", Solution().getMinimumDifference(deep))
assert Solution().getMinimumDifference(deep) == 2

400 BST ngẫu nhiên: 7 cài đặt khớp brute force mọi cặp — PASS


cây lệch 10^4 node: 2


## 13. Lỗi người mới hay gặp

### Lỗi 1: `if prev:` thay vì `if prev is not None:`
Khi `prev = 0` (node giá trị 0), điều kiện sai → bỏ qua một cặp.

### Lỗi 2: chỉ so cha–con
Đã có phản ví dụ ở mục 7.

### Lỗi 3: dùng pre-order với `prev`
Pre-order không cho dãy tăng → so sai cặp, hiệu có thể âm.

### Lỗi 4: khởi tạo `best = 0`
`min(0, ...)` luôn là 0 → sai. Khởi tạo bằng `float('inf')`.

In [17]:
def buggy_truthy_prev(root):
    vals = inorder_values(root, [])
    prev, best = None, float("inf")
    for v in vals:
        if prev:                       # sai khi prev == 0
            best = min(best, v - prev)
        prev = v
    return best

def buggy_preorder_prev(root):
    vals = preorder_values(root, [])
    prev, best = None, float("inf")
    for v in vals:
        if prev is not None:
            best = min(best, v - prev)  # không abs, và không phải dãy tăng
        prev = v
    return best

t0 = build_tree([2, 0, 9])              # cặp (0, 2) = 2 là nhỏ nhất
print("if prev: ->", buggy_truthy_prev(t0), "| đúng:", Solution().getMinimumDifference(t0))
print("pre-order ->", buggy_preorder_prev(build_tree([4, 2, 6, 1, 3])), "| đúng:", 1)

if prev: -> 7 | đúng: 2
pre-order -> -2 | đúng: 1


Với `[2, 0, 9]`: in-order `0, 2, 9`. `if prev:` bỏ qua cặp `(0, 2)` vì `prev = 0` là falsy → trả `7` thay vì `2`. Bản pre-order (dãy `4, 2, 1, 3, 6`) cho cả hiệu âm (`-2`)!

## 14. Mẹo nhận dạng pattern

- **BST** + "nhỏ nhất/lớn nhất giữa hai giá trị", "phần tử thứ k", "kiểm tra hợp lệ", "đếm trùng" → **in-order = dãy sắp xếp** + biến `prev`.
- "Hiệu nhỏ nhất giữa hai phần tử bất kỳ" trên dãy → **sắp xếp rồi so phần tử kề nhau**.
- Không chắc "chỉ cần so cha–con"? Hãy tìm phản ví dụ bằng **test đối chiếu ngẫu nhiên** với brute force.
- Cây có thể sâu tới 10⁴ → ưu tiên **duyệt lặp bằng stack**.

## 15. Biến thể

### 15.1. LeetCode 783 — Minimum Distance Between BST Nodes
Giống hệt: dùng nguyên lời giải.

### 15.2. Mảng thường — Minimum Absolute Difference (LeetCode 1200)
Trả về **mọi cặp** có hiệu nhỏ nhất (theo thứ tự tăng).

In [18]:
def minimum_abs_difference_pairs(arr):
    arr = sorted(arr)
    best = min(b - a for a, b in zip(arr, arr[1:]))
    return [[a, b] for a, b in zip(arr, arr[1:]) if b - a == best]

print(minimum_abs_difference_pairs([4, 2, 1, 3]))           # [[1,2],[2,3],[3,4]]
print(minimum_abs_difference_pairs([3, 8, -10, 23, 19, -4, -14, 27]))

[[1, 2], [2, 3], [3, 4]]
[[-14, -10], [19, 23], [23, 27]]


### 15.3. Kiểm tra BST hợp lệ bằng in-order + `prev`
Cùng khung code: chỉ đổi "cập nhật best" thành "nếu `node.val <= prev` thì không hợp lệ".

In [19]:
def is_valid_bst(root):
    stack, cur, prev = [], root, None
    while stack or cur:
        while cur:
            stack.append(cur); cur = cur.left
        cur = stack.pop()
        if prev is not None and cur.val <= prev:
            return False
        prev = cur.val
        cur = cur.right
    return True

print(is_valid_bst(build_tree([4, 2, 6, 1, 3])), is_valid_bst(build_tree([5, 1, 4, None, None, 3, 6])))

True False


## 16. Bài tập (đáp án ở cuối)

### Bài 1 — Tính tay
Cho BST `[27, 10, 50, 5, 20, 40, 60, None, 8, 15]`. Viết dãy in-order, các hiệu kề nhau, và đáp án.

### Bài 2 — Điền TODO (in-order lặp)

In [20]:
def exercise_min_diff(root):
    stack, cur = [], root
    prev, best = None, float("inf")
    while stack or cur:
        # TODO 1: đi trái hết mức, đẩy vào stack
        # TODO 2: pop, so với prev (nhớ prev có thể là 0), cập nhật prev
        # TODO 3: chuyển sang con phải
        pass
    return best

# assert exercise_min_diff(build_tree([4, 2, 6, 1, 3])) == 1

### Bài 3 — Hiệu **lớn nhất** giữa hai node của BST
Gợi ý: không cần duyệt hết cây.

In [21]:
def max_diff_bst(root):
    # TODO
    pass

# assert max_diff_bst(build_tree([1, 0, 48, None, None, 12, 49])) == 49

### Bài 4 — Trả về **cặp** giá trị có hiệu nhỏ nhất
Trả tuple `(a, b)` với `a < b`, nếu nhiều cặp thì lấy cặp xuất hiện đầu tiên trong in-order.

In [22]:
def closest_pair_bst(root):
    # TODO
    pass

# assert closest_pair_bst(build_tree([1, 0, 48, None, None, 12, 49])) == (0, 1)

### Bài 5 — Cây nhị phân **thường** (không phải BST)
Trả về hiệu tuyệt đối nhỏ nhất giữa hai node bất kỳ.

In [23]:
def min_diff_any_tree(root):
    # TODO
    pass

# assert min_diff_any_tree(build_tree([10, 50, 3, 49, None, 1])) == 1

### Bài 6 — Mini challenge: phần tử nhỏ thứ k trong BST (LeetCode 230)
Dùng in-order lặp, dừng ngay khi thăm đủ `k` node.

In [24]:
def kth_smallest(root, k):
    # TODO
    pass

# assert kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3) == 3

## 17. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Vẽ cây theo tầng | in-order: trái → node → phải | kiểm tra bằng `trace_min_diff(build_tree(...))` |
| 2 | Xem mục 6.2 | `while cur: stack.append(cur); cur = cur.left` | `if prev is not None: best = min(best, cur.val - prev)` |
| 3 | Lớn nhất − nhỏ nhất | nhỏ nhất ở node trái cùng, lớn nhất ở node phải cùng | hai vòng `while node.left` / `while node.right` |
| 4 | Như lời giải chính | lưu thêm `pair` khi `best` giảm | chỉ cập nhật khi `diff < best` (dấu `<` giữ cặp đầu tiên) |
| 5 | In-order không còn sắp xếp | thu giá trị rồi `sorted` | so phần tử kề nhau |
| 6 | Đếm số node đã thăm | `k -= 1` sau mỗi lần pop | khi `k == 0` trả `cur.val` |

## 18. Đáp án tham khảo

**Bài 1:** in-order `5, 8, 10, 15, 20, 27, 40, 50, 60`; hiệu kề nhau `3, 2, 5, 5, 7, 13, 10, 10` → đáp án **2** (cặp `8, 10`).

In [25]:
t1 = build_tree([27, 10, 50, 5, 20, 40, 60, None, 8, 15])
show_tree(t1)
print("in-order:", inorder_values(t1, []))
trace_min_diff(t1)

          60
     50
          40
27
          20
               15
     10
               8
          5
in-order: [5, 8, 10, 15, 20, 27, 40, 50, 60]
bước 1: stack=[27, 10, 5] thăm   5 | prev=None | hiệu=—   best=inf
bước 2: stack=[27, 10, 8] thăm   8 | prev=   5 | hiệu=3   best=3
bước 3: stack=[27, 10]   thăm  10 | prev=   8 | hiệu=2   best=2
bước 4: stack=[27, 20, 15] thăm  15 | prev=  10 | hiệu=5   best=2
bước 5: stack=[27, 20]   thăm  20 | prev=  15 | hiệu=5   best=2
bước 6: stack=[27]       thăm  27 | prev=  20 | hiệu=7   best=2
bước 7: stack=[50, 40]   thăm  40 | prev=  27 | hiệu=13  best=2
bước 8: stack=[50]       thăm  50 | prev=  40 | hiệu=10  best=2
bước 9: stack=[60]       thăm  60 | prev=  50 | hiệu=10  best=2


2

In [26]:
def answer_exercise_min_diff(root):
    stack, cur = [], root
    prev, best = None, float("inf")
    while stack or cur:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        if prev is not None:
            best = min(best, cur.val - prev)
        prev = cur.val
        cur = cur.right
    return best

def answer_max_diff_bst(root):
    lo = hi = root
    while lo.left:
        lo = lo.left
    while hi.right:
        hi = hi.right
    return hi.val - lo.val

def answer_closest_pair_bst(root):
    vals = inorder_values(root, [])
    best, pair = float("inf"), None
    for a, b2 in zip(vals, vals[1:]):
        if b2 - a < best:
            best, pair = b2 - a, (a, b2)
    return pair

def answer_min_diff_any_tree(root):
    vals = sorted(preorder_values(root, []))
    return min(b2 - a for a, b2 in zip(vals, vals[1:]))

def answer_kth_smallest(root, k):
    stack, cur = [], root
    while stack or cur:
        while cur:
            stack.append(cur)
            cur = cur.left
        cur = stack.pop()
        k -= 1
        if k == 0:
            return cur.val
        cur = cur.right

assert answer_exercise_min_diff(build_tree([4, 2, 6, 1, 3])) == 1
assert answer_exercise_min_diff(t1) == 2
assert answer_max_diff_bst(build_tree([1, 0, 48, None, None, 12, 49])) == 49
assert answer_closest_pair_bst(build_tree([1, 0, 48, None, None, 12, 49])) == (0, 1)
assert answer_min_diff_any_tree(build_tree([10, 50, 3, 49, None, 1])) == 1
assert answer_kth_smallest(build_tree([5, 3, 6, 2, 4, None, None, 1]), 3) == 3
print("Đáp án bài 2–6: PASS toàn bộ test!")

Đáp án bài 2–6: PASS toàn bộ test!


## 19. Template nhớ nhanh + checklist submit

```python
stack, cur = [], root
prev, best = None, float("inf")
while stack or cur:
    while cur:                       # đi trái hết mức
        stack.append(cur); cur = cur.left
    cur = stack.pop()                # thăm theo thứ tự tăng dần
    if prev is not None:
        best = min(best, cur.val - prev)
    prev = cur.val
    cur = cur.right                  # sang phải
return best
```

Câu nhớ: **"BST → in-order tăng dần → đáp án nằm giữa hai phần tử kề nhau."**

### Checklist
- [ ] Duyệt **in-order**, so với `prev` (node thăm ngay trước).
- [ ] `prev is not None` (giá trị 0 là hợp lệ).
- [ ] `best` khởi tạo `float('inf')`.
- [ ] Không chỉ so cha–con.
- [ ] Cây sâu 10⁴ → dùng bản lặp.
- [ ] Time `O(n)`, Space `O(h)`.